# Transfer learning - MobileNetV2 (ImageNet)

In [ ]:
import shutil
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from results_utils import DATA_DIR, MODELS_DIR, RESULTS_DIR, callbacks_paths, save_results

tf.keras.utils.set_random_seed(42)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

GRID_LR = [1e-3, 1e-4]
GRID_DROPOUT = [0.2, 0.5]
DENSE_UNITS = 128
GRID_EPOCHS = 10
GRID_PATIENCE = 3

FINE_TUNE_LAYERS = 30
FINE_TUNE_LR = 1e-5
FINE_TUNE_EPOCHS = 10
FINE_TUNE_PATIENCE = 4

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "valid",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=42).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

counts = np.array([len(list((DATA_DIR / "train" / c).glob("*.jpg"))) for c in class_names])
class_weight = {i: counts.sum() / (len(counts) * n) for i, n in enumerate(counts)}

In [ ]:
def build_tl_model(dropout=0.2, dense_units=128, num_classes=10, input_shape=(224, 224, 3)):
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.2),
    ], name="data_augmentation")

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./127.5, offset=-1)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(dense_units, activation='relu', kernel_initializer='he_uniform')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name="mobilenetv2_tl"), base_model

## 1. Feature extraction (frozen base) - grid search on learning rate x dropout

In [ ]:
grid_results = []
grid_models = {}
(RESULTS_DIR / "grid").mkdir(exist_ok=True)
start = time.time()

for lr in GRID_LR:
    for dropout in GRID_DROPOUT:
        name = f"lr{lr}_do{dropout}"
        print(f"\n===== {name} =====")
        tf.keras.utils.set_random_seed(42)

        model, base_model = build_tl_model(dropout=dropout, dense_units=DENSE_UNITS, num_classes=len(class_names))
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

        h = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=GRID_EPOCHS,
            class_weight=class_weight,
            callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=GRID_PATIENCE, restore_best_weights=True),
                       callbacks.CSVLogger(RESULTS_DIR / "grid" / f"{name}_history.csv")],
            verbose=2
        )

        best = int(np.argmin(h.history["val_loss"]))
        grid_results.append({"config": name, "lr": lr, "dropout": dropout, "epochs": len(h.history["loss"]),
                             "train_acc": h.history["accuracy"][best], "val_acc": h.history["val_accuracy"][best],
                             "val_loss": h.history["val_loss"][best]})
        grid_models[name] = (model, base_model)

print(f"Grid search time: {(time.time() - start) / 60:.1f} min")

grid = pd.DataFrame(grid_results).sort_values("val_loss").reset_index(drop=True)
grid.to_csv(RESULTS_DIR / "tl_grid_search.csv", index=False)
grid

In [ ]:
best_cfg = grid.iloc[0]
print(f"Best config (lowest val_loss): lr={best_cfg['lr']}, dropout={best_cfg['dropout']}")

model, base_model = grid_models[best_cfg["config"]]
MODEL_NAME = "tl_mobilenetv2_frozen"
best_model_path, history_path = callbacks_paths(MODEL_NAME)
model.save(best_model_path)
shutil.copy(RESULTS_DIR / "grid" / f"{best_cfg['config']}_history.csv", history_path)

params = {"epochs": GRID_EPOCHS, "lr": best_cfg["lr"], "patience": GRID_PATIENCE, "batch_size": BATCH_SIZE,
          "filters": "MobileNetV2 frozen", "dropout": best_cfg["dropout"], "dense_units": DENSE_UNITS,
          "augment": True, "batchnorm": True, "class_weight": True}

_, _, _, frozen_acc = save_results(MODEL_NAME, model, test_ds, class_names, params)
print(f"Test accuracy (frozen): {frozen_acc:.2%}")

## 2. Fine-tuning the last layers of MobileNetV2

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

print(f"Trainable layers in the base: {sum(l.trainable for l in base_model.layers)} / {len(base_model.layers)}")

model.compile(optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

MODEL_NAME = "tl_mobilenetv2_finetuned"
best_model_path, history_path = callbacks_paths(MODEL_NAME)
start = time.time()

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weight,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=FINE_TUNE_PATIENCE, restore_best_weights=True),
               callbacks.ModelCheckpoint(best_model_path, monitor="val_loss", save_best_only=True),
               callbacks.CSVLogger(history_path)],
    verbose=2
)

print(f"Fine-tuning time: {(time.time() - start) / 60:.1f} min")

In [ ]:
params = {"epochs": FINE_TUNE_EPOCHS, "lr": FINE_TUNE_LR, "patience": FINE_TUNE_PATIENCE, "batch_size": BATCH_SIZE,
          "filters": f"MobileNetV2 last {FINE_TUNE_LAYERS} layers", "dropout": best_cfg["dropout"], "dense_units": DENSE_UNITS,
          "augment": True, "batchnorm": True, "class_weight": True}

true_indices, predicted_indices, predictions, test_acc = save_results(MODEL_NAME, model, test_ds, class_names, params)
print(f"Test accuracy: frozen {frozen_acc:.2%} -> fine-tuned {test_acc:.2%}")

## 3. Results

In [ ]:
frozen_hist = pd.read_csv(RESULTS_DIR / "tl_mobilenetv2_frozen_history.csv")
ft_hist = pd.read_csv(history_path)
hist = pd.concat([frozen_hist, ft_hist], ignore_index=True)
start_ft = len(frozen_hist) - 0.5

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist["accuracy"], label='Train Accuracy')
plt.plot(hist["val_accuracy"], label='Val Accuracy')
plt.axvline(start_ft, color='gray', linestyle='--', label='Start fine-tuning')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(hist["loss"], label='Train Loss')
plt.plot(hist["val_loss"], label='Val Loss')
plt.axvline(start_ft, color='gray', linestyle='--', label='Start fine-tuning')
plt.legend()
plt.title('Loss')
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
for name in grid["config"]:
    h = pd.read_csv(RESULTS_DIR / "grid" / f"{name}_history.csv")
    plt.subplot(1, 2, 1)
    plt.plot(h["val_accuracy"], label=name)
    plt.subplot(1, 2, 2)
    plt.plot(h["val_loss"], label=name)

plt.subplot(1, 2, 1)
plt.title("Grid search - Val Accuracy")
plt.legend()
plt.subplot(1, 2, 2)
plt.title("Grid search - Val Loss")
plt.legend()
plt.show()

In [ ]:
cm = confusion_matrix(true_indices, predicted_indices)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title(f"{MODEL_NAME} - test accuracy {test_acc:.0%}")
plt.tight_layout()
plt.show()

In [ ]:
test_images = np.concatenate([images.numpy() for images, _ in test_ds], axis=0)
confidences = predictions.max(axis=1)
wrong = np.where(predicted_indices != true_indices)[0]
rows = max(1, int(np.ceil(len(wrong) / 5)))

fig, axes = plt.subplots(rows, 5, figsize=(16, 3.5 * rows), squeeze=False)
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, i in zip(axes, wrong):
    ax.imshow(test_images[i].astype("uint8"))
    ax.set_title(f"Pred: {class_names[predicted_indices[i]]}\n(True: {class_names[true_indices[i]]})\n{confidences[i] * 100:.0f}%", color="red", fontsize=9)

plt.suptitle(f"Misclassified test images: {len(wrong)} / {len(true_indices)}")
plt.tight_layout()
plt.show()